Defining functions and packages for this pipeline

In [ ]:
import os
import re
from pathlib import Path
from IPython.display import display
import tifffile
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
import warnings
import cellpose
from cellpose import models
from scipy.ndimage import zoom
from skimage.measure import regionprops, regionprops_table
import esda
import libpysal
import torch
import seaborn as sns
import matplotlib.pyplot as plt

# Function to validate and process the directory
def process_directory(input_dir):
    pathI = Path(input_dir)

    if not pathI.exists() or not pathI.is_dir():
        raise FileNotFoundError(f"The specified path does not exist or is not a directory: {pathI}")

    files = [f.name for f in pathI.iterdir() if f.is_file()]
    if not files:
        raise FileNotFoundError(f"No files found in the directory: {pathI}")

    files.insert(0, "None")
    return pathI, files

# Function to create and display widgets for user input
def create_widgets(files, pathI):
    user_data = {}  # Dictionary to store user data

    projXY_selector = widgets.Dropdown(
        options=files,
        description='projXY:',
        layout=widgets.Layout(width='50%'),
        style={'description_width': 'initial'}
    )

    projXZ_selector = widgets.Dropdown(
        options=files,
        description='projXZ:',
        layout=widgets.Layout(width='50%'),
        style={'description_width': 'initial'}
    )

    crop_check = widgets.Checkbox(
        value=False,
        description='Apply crop?',
        style={'description_width': 'initial'}
    )

    output_name = widgets.Text(
        value='output',
        placeholder='Enter output file name',
        description='Output Name:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    submit_button = widgets.Button(description="Submit")
    output = widgets.Output()

    # Callback function
    def select_files_callback(b):
        projXY_file = projXY_selector.value
        projXZ_file = projXZ_selector.value
        projXY_path, projXZ_path = None, None

        with output:
            output.clear_output()
            if projXY_file != "None":
                projXY_path = pathI / projXY_file
                if os.path.exists(projXY_path):
                    print(f"Selected projXY file: {projXY_path}")
                    # Get dimensions without opening the image
                    with tifffile.TiffFile(projXY_path) as tif:
                        projXY_dims = tif.pages[0].shape  # (height, width)
                    print(f"projXY dimensions: {projXY_dims}")
                else:
                    print(f"Warning: {projXY_path} does not exist.")

            if projXZ_file != "None":
                projXZ_path = pathI / projXZ_file
                if os.path.exists(projXZ_path):
                    print(f"Selected projXZ file: {projXZ_path}")
                    # Get dimensions without opening the image
                    with tifffile.TiffFile(projXZ_path) as tif:
                        projXZ_dims = tif.pages[0].shape  # (height, width)
                    print(f"projXZ dimensions: {projXZ_dims}")
                else:
                    print(f"Warning: {projXZ_path} does not exist.")

            user_data.update({
                'projXY_path': projXY_path,
                'projXZ_path': projXZ_path,
                'crop': crop_check.value,
                'output_name': output_name.value,
            })

            print("User data updated:", user_data)

    # Connect the callback to the button
    submit_button.on_click(select_files_callback)

    # Show the widgets
    display(projXY_selector, projXZ_selector, crop_check, output_name, submit_button, output)

    # Return the collected data once the button is clicked
    return user_data  # Return the collected data

# Function to crop TIFF stack based on max projections made in FIJI
def crop_tiff_stack(input_dir, output_dir, projXY_path, projXZ_path, crop=True):
    try:
        # Load projections
        projXY = tifffile.imread(projXY_path)
        projXZ = tifffile.imread(projXZ_path)

        # Ensure output directory exists
        os.makedirs(output_dir, exist_ok=True)

        # Regex to find frame info
        pattern = re.compile(r'frame-(\d+)\.tif')

        # Find available frames
        frame_numbers = set()
        for file in os.listdir(input_dir):
            match = pattern.search(file)
            if match:
                frame_numbers.add(int(match.group(1)))

        if not frame_numbers:
            raise ValueError("No frames found in the input directory.")

        frames = sorted(frame_numbers)

        for t in frames:
            ref_path = os.path.join(input_dir, f"Channel-ref-frame-{t}.tif")
            crc_path = os.path.join(input_dir, f"Channel-CRC-frame-{t}.tif")
            wt_path = os.path.join(input_dir, f"Channel-WT-frame-{t}.tif")
            
            if os.path.exists(ref_path):
                print(f"Processing frame {t}")

                if crop:
                    frame = tifffile.imread(ref_path)
                    print(f"Original shape: {frame.shape}")

                    # Determine the number of channels
                    if frame.ndim == 4:
                        num_channels = frame.shape[1]
                    else:
                        num_channels = 1
                        frame = frame[:, np.newaxis, :, :]

                    # Initialize cropping limits
                    slice_min, slice_max = 0, frame.shape[0]
                    row_min, row_max = 0, frame.shape[2]
                    col_min, col_max = 0, frame.shape[3]

                    # Create masks based on projections
                    maskXY = projXY[t] > 0
                    maskXZ = projXZ[t] > 0

                    # Find coordinates of non-zero pixels
                    coordsXY = np.where(maskXY)
                    coordsXZ = np.where(maskXZ)

                    # Adjust cropping limits based on XY projection
                    if coordsXY[0].size > 0:
                        row_min, row_max = np.min(coordsXY[0]), np.max(coordsXY[0]) + 1
                        col_min, col_max = np.min(coordsXY[1]), np.max(coordsXY[1]) + 1

                    # Adjust cropping limits based on XZ projection
                    if coordsXZ[0].size > 0:
                        slice_min, slice_max = np.min(coordsXZ[0]), np.max(coordsXZ[0]) + 1

                    # Process each channel
                    for channel_path, channel_name in zip([ref_path, crc_path, wt_path], ["ref", "CRC", "WT"]):
                        if os.path.exists(channel_path):
                            frame = tifffile.imread(channel_path)
                            if frame.ndim == 3:
                                frame = frame[:, np.newaxis, :, :]
                            
                            cropped = frame[slice_min:slice_max, :, row_min:row_max, col_min:col_max]
                            print(f"Cropped shape ({channel_name}): {cropped.shape}")

                            # Ensure the correct shape: (T, Z, C, Y, X, S)
                            cropped = np.expand_dims(cropped, axis=0)  # Add T dimension
                            cropped = np.expand_dims(cropped, axis=-1) # Add S dimension
                            
                            output_path = os.path.join(output_dir, f"Channel-{channel_name}-frame-{t}.tif")
                            tifffile.imwrite(
                                output_path,
                                cropped,
                                imagej=True,
                                metadata={'axes': 'TZCYXS'},
                            )
                        else:
                            print(f"Warning: {channel_path} not found.")
            else:
                print(f"Frame {t} not found: {ref_path}")

        print(f"Frames cropped and saved to {output_dir}")

    except FileNotFoundError as e:
        print(f"Error: {e}")
    except ValueError as e:
        print(f"Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Function to compute K neirest neighbours features on nuclei segmentation masks made by cellpose
def compute_knn_features(points, k=5):
    if len(points) == 0:
        return [], [], []

    setX = points['x'] / points['x'].max()
    setY = points['y'] / points['y'].max()
    setZ = points['z'] / points['z'].max()

    coords = np.stack([setX, setY, setZ], axis=1)

    tree = KDTree(coords)
    k_use = min(k, len(coords))
    distances, indices = tree.query(coords, k=k_use)

    distAv = []
    pWT = []
    pC = []

    for i in range(len(indices)):
        neighbor_indices = indices[i]
        neighbor_phenotypes = points.iloc[neighbor_indices]['Phenotype']

        count_WT = sum(p.startswith('WT') for p in neighbor_phenotypes)
        count_C = sum(p.startswith('C') for p in neighbor_phenotypes)
        total = len(neighbor_phenotypes)

        distAv.append(np.median(distances[i]))
        pWT.append(count_WT)
        pC.append(count_C)

    return distAv, pWT, pC


Crop all frames based on the max projections

In [ ]:
# Fill in the input directory where the TIFF files are located
input_dir = r""

try:
    pathI, files = process_directory(input_dir)
    user_data = create_widgets(files, pathI)  
    print(user_data)      
except Exception as e:
    print(f"Error: {e}")

output_directory = os.path.join(str(input_dir), user_data['output_name'])
crop_tiff_stack(input_dir, output_directory, user_data['projXY_path'], user_data['projXZ_path'])

Run cellpose on each slice and frame and use cellpose.Stitch3D to create 3D nuclei masks

In [ ]:
input_directory = r"Z:\users\6331823\Mario Pipeline\frames-frames_extracted\Second try" # All the crops you generated above
output_directory = os.path.join(input_directory, "cellpose")
rescale_factor = 0.5  # Zoom factor that is used before running cellpose to speed up the segmentation
custom_model = False # False to use cellpose SAM, true to use custom model
custom_model_path = r""  # Path to custom model if custo_model is True

# Make the output directory if it does not exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Check if there is a GPU for cellpose, will significantly speed up the segmentation
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device.upper()} for processing.")

# Load the cellpose model
if custom_model == True:
    if os.path.exists(custom_model_path):
        model = models.CellposeModel(gpu=(device == "cuda"), pretrained_model=custom_model_path)
        print(f"Using custom model: {custom_model_path}")
    else:
        warnings.warn("Custom model not found. Using the default nuclei model.")
        model = models.Cellpose(gpu=(device == "cuda"), model_type="nuclei")
else:
    model = models.CellposeModel(gpu=(device == "cuda"))
    print("Using SAM standard cellpose model")

# Find all nuclear staining TIFF crops you made previously but 
files = sorted([f for f in os.listdir(input_directory) if f.endswith(".tif") and f.startswith("Channel-ref-")])

summary_results = []

for file in files:
    frame_path = os.path.join(input_directory, file)
    print(f"Processing {file}")

    frame = tifffile.imread(frame_path)
    
    # Make sure the frame only has 1 channel data, if not, take only the first channel
    if frame.ndim == 4:  # Multi-channel case (Z, C, Y, X)
        frame = frame[:, 0, :, :]  
    else:
        frame = frame  
    
    depthIm, heightIm, widthIm = frame.shape
    original_shape =  (depthIm, heightIm, widthIm) # Save original shape to later rescale the masks back to original size
    
    frame = zoom(frame, zoom = rescale_factor, order=1) # Rescale image in all 3 dimensions to imporove speed
    
    depthIm, heightIm, widthIm = frame.shape
              
    # Threshold the image to remove low-intensity noise
    mean_intensity = np.mean(frame)
    std_intensity = np.std(frame)
    threshold = mean_intensity + std_intensity
    frame[frame < threshold] = 0
    
    # Parameters to calculate Moran's I later on, which judges spatial autocorrelation of signal (real vs background signal)
    shapeW = zoom(frame, zoom = (1,0.5,0.5), order=1)
    w = libpysal.weights.lat2W(shapeW.shape[1], shapeW.shape[2])
    
    masks_list = [] # Save masks of every slice for this frame to later stitch them into a 3D mask
    
    # Go over each z-slice, evaluate whether this slice contains background signal or a real organoid based on Moran's I
    # If it is a real signal slice, run cellpose to segment the nuclei, if not, fill the mask with zeros/no cells found
    for z in range(depthIm):
        plane_2d = frame[z, :, :]
        planeMoran = shapeW[z, :, :]
        moran = esda.Moran(planeMoran.ravel(), w)
        if moran.I > 0.6:
            print(f"Slice {z}, moran: {moran.I}")
            masks, _, _ = model.eval(plane_2d, diameter=None, do_3D=False)             
        else:
            masks = np.zeros_like(plane_2d, dtype=np.uint16)
            print(f"Skipping slice {z} (low signal), moran: {moran.I}")
            
        masks_list.append(masks)

    # Stitch the masks from all slices into a 3D mask
    full_mask_3d = np.stack(masks_list, axis=0).astype(np.uint16)
    full_mask_3d = cellpose.utils.stitch3D(full_mask_3d)
    
    # Rescale the mask back to the orignal size
    zoom_factors = np.array(original_shape) / np.array(full_mask_3d.shape)
    mask_original = zoom(full_mask_3d, zoom=zoom_factors, order=0).astype(np.uint16)    
        
    # Save the 3D segmentation mask as a TIFF file
    segmented_tiff_file = os.path.join(output_directory, f"{file.split('.')[0]}_segmented.tif")
    tifffile.imwrite(segmented_tiff_file, mask_original)
    print(f"Segmented masks saved: {segmented_tiff_file}")
    
    # Get the properties of all the nuclei, which will be used to determine phenotypes and calculate KNN features
    props = regionprops(mask_original)
    data = []
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area

        data.append({
            'Label': label,
            'z': centroid[0],
            'y': centroid[1],
            'x': centroid[2],
            'BoundingBox': bounding_box,
            'Volume': volume
        })

    # Save these properties to a text file
    df_props = pd.DataFrame(data).rename(columns={'Label': 'label'})
    output_txt_file = os.path.join(output_directory, f"{file.split('.')[0]}_props.txt")
    df_props.to_csv(output_txt_file, sep='\t', index=False)
    print(f"Properties saved to: {output_txt_file}")   

    # Match the segmentated frame with the channels of the TIFF crops for WT and CRC channel
    frameNumber = re.search(r'frame-(\d+)', file).group(1)
    ch_wt = tifffile.imread(os.path.join(input_directory, f"Channel-WT-frame-{frameNumber}.tif"))
    ch_crc = tifffile.imread(os.path.join(input_directory, f"Channel-CRC-frame-{frameNumber}.tif"))

    props_wt = regionprops_table(mask_original, intensity_image=ch_wt,
                                 properties=['label', 'area', 'mean_intensity'])
    props_crc = regionprops_table(mask_original, intensity_image=ch_crc,
                                  properties=['label', 'area', 'mean_intensity'])

    df_wt = pd.DataFrame(props_wt).rename(columns={'area': 'Area_WT', 'mean_intensity': 'Raw_WT'})
    df_crc = pd.DataFrame(props_crc).rename(columns={'area': 'Area_CRC', 'mean_intensity': 'Raw_CRC'})

    # Combinar todo
    df_final = df_props.merge(df_wt, on='label').merge(df_crc, on='label')
    
    # Assign phentoypes according to the defined thresholds
    phenotype = []
    count_phenotype_c1 = 0
    count_phenotype_c2 = 0
    count_phenotype_c1and2 = 0
    count_phenotype_empty = 0
    k_channel1 = 0
    k_channel2 = 0.5 

    for _, row in df_final.iterrows():
        VolTot = row['Volume']
        VolCh1 = row['Area_WT']
        avgCh1 = row['Raw_WT'] / VolCh1 if VolCh1 > 0 else 0
        VolCh2 = row['Area_CRC']
        avgCh2 = row['Raw_CRC'] / VolCh2 if VolCh2 > 0 else 0

        pheno = "-"
        qa = 0
        qb = 0

        if VolCh1 > k_channel1 * VolTot:
            # Fenotipo WT
            pheno_c1 = "WT"
            count_phenotype_c1 += 1
            qa = 1
        else:
            pheno_c1 = "-"

        if VolCh2 > k_channel2 * VolTot:
            # Fenotipo CRC
            pheno_c2 = "CRC"
            pheno = "CRC"
            count_phenotype_c2 += 1
            qb = 1
        else:
            pheno_c2 = "-"

        if qa + qb == 2:
            if avgCh1 >= 2 * avgCh2:
                pheno = "WT"
            else:
                pheno = "CRC"
            count_phenotype_c1and2 += 1
        elif pheno == "-":
            count_phenotype_empty += 1

        if qa == 1 and qb == 0:
            pheno = "WT"

        phenotype.append(pheno)

    df_final['Phenotype'] = phenotype
    
    # Save the quantified data to a CSV file
    output_csv = os.path.join(output_directory, f"{file.split('.')[0]}_quantified.csv")
    df_final.to_csv(output_csv, index=False)
    print(f"Quantified data saved to: {output_csv}")
    
    # Append data from this frame to the summary
    summary_results.append({
        'file': file,
        'phenotype_count_c1': count_phenotype_c1,
        'phenotype_count_c2': count_phenotype_c2,
        'phenotype_count_c1and2': count_phenotype_c1and2,
        'phenotype_count_empty': count_phenotype_empty
    })  

# Save summary of the whole movie as a csv
summary_results = pd.DataFrame(summary_results)
summary_csv = os.path.join(output_directory, "summary_results.csv")
summary_results.to_csv(summary_csv, index=False)

Summarise the whole TL

In [9]:
# Set summary file input folder
input_dir = r"Z:\users\6331823\Mario Pipeline\frames-frames_extracted\Second try\cellpose"
output_path = os.path.join(input_dir, "distances")
TARGET_SERIES = "TL09" # Change this to the series you want to analyze

os.makedirs(output_path, exist_ok=True)

# Load the results
dataFolder = pd.read_csv(os.path.join(input_dir, "summary_results.csv"), sep=',')

# Pattern to recognize the series_id and frame number in the summary data 
# pattern = r'^(?P<series_id>.+?)-frame-(?P<frame>\d+)\.txt$' #the correct one
pattern = r'^(?P<series_id>.+?)-ref-frame-(?P<frame>\d+)\.tif$' #altered for test
dataFolder[['series_id', 'frame']] = dataFolder['file'].str.extract(pattern)

# change frame numbers from strings to integers
dataFolder['frame'] = dataFolder['frame'].astype(pd.Int64Dtype())

# This for loop find all frames and compute phenotype ratios and KNN features
summary_stats = []
for sid in dataFolder['series_id'].unique():
    subset = dataFolder[dataFolder['series_id'] == sid].sort_values(by='frame') # Gets a list of all TL and their frames
    databyTL = pd.DataFrame()  # make DF to store data for this TL

    for _, row in subset.iterrows(): # Goes over each frame of one TL
        fileN = "Channel-ref-frame-" + str(row['frame']) + "_quantified.csv"
        series_id = row['series_id']
        frame = row['frame']
        filePath = os.path.join(input_dir, fileN) # This file are all the df_final statistics per frame we made earlier, containing phenotype data
        
        # Read the data per frame
        try:
            oscarDATA = pd.read_csv(filePath, sep=',') 
        except Exception as e:
            print(f"Summary non existent {fileN}: {e}")
            continue
        
        # i check if the spatial cooridnates x,y,z (named as cx, cy,z by OSCAR in my case) are present, but you can change the names and verify the columns you need
        if not all(col in oscarDATA.columns for col in ['x', 'y', 'z', 'Phenotype']):
            print(f"Columns are missing in {fileN}")
            continue

        # Create a place to store statistics
        oscarDATA['frame'] = frame
        oscarDATA['series_id'] = series_id

        # Generate statistics on phenotypes per frame
        totalCells = len(oscarDATA)
        pointsWT = oscarDATA[oscarDATA['Phenotype'].str.startswith("WT")].copy()
        pointsC = oscarDATA[oscarDATA['Phenotype'].str.startswith("C")].copy()

        # Calculate KNN features
        distAv, pWT, pC = compute_knn_features(oscarDATA)
        oscarDATA['K-NN(k=5)'] = distAv
        oscarDATA['Num_WT'] = pWT
        oscarDATA['Num_C'] = pC

        # Append data of this frame to summary statistics
        summary_stats.append({
            'file': fileN,
            'series_id': series_id,
            'frame': frame,
            'total_cells': totalCells,
            'WT_cells': len(pointsWT),
            'C_cells': len(pointsC),
            'median_knn_dist': np.median(distAv) if len(distAv) > 0 else np.nan,
            'mean_Num_WT': np.mean(pWT) if len(pWT) > 0 else np.nan,
            'mean_Num_C': np.mean(pC) if len(pC) > 0 else np.nan
        })

        databyTL = pd.concat([databyTL, oscarDATA], ignore_index=True)

    # Save the data of each TL to a txt file
    temp_path = os.path.join(output_path, f"{sid}_df.txt")
    databyTL.to_csv(temp_path, sep='\t', index=False)   
        
    # #
    #if sid == TARGET_SERIES:
    max_frame =   int(databyTL['frame'].max()) + 1
    max_z = int(  databyTL['z'].max()) + 1
    max_y = int(  databyTL['y'].max()) + 1
    max_x = int(  databyTL['x'].max()) + 1
    #
    array_5d = np.zeros((max_frame, 3, max_z, max_y, max_x), dtype=np.uint8)
    for tt in range(max_frame):
        for _, row in databyTL.iterrows():
            x, y, z = int(row['x']), int(row['y']), int(row['z'])
            frame = int(row['frame'])
            phenotype = row['Phenotype']
            array_5d[frame, 0, z, y, x] = 255  # Total
            if phenotype.startswith("WT"):
                array_5d[frame, 1, z, y, x] = 255
            elif phenotype.startswith("C"):
                array_5d[frame, 2, z, y, x] = 255

    # Reorganizar ejes a (T, Z, C, Y, X)
    array_5d = np.transpose(array_5d, (0, 2, 1, 3, 4))

    # Guardar como TIFF con metadatos para ImageJ
    tiff_path = os.path.join(output_path, f"{sid}_stack_TZCYX.tiff")
    tifffile.imwrite(
        tiff_path,
        array_5d,
        photometric='minisblack',
        metadata={
            'axes': 'TZCYX',  # Especificar orden de ejes
            'spacing': 1,     # Opcional: ajustar según calibración real
            'unit': 'um'      # Opcional: unidades físicas
        },
        imagej=True
    )
    print(f"✅ TIFF guardado correctamente: {tiff_path}")

# Guardar resumen total
summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(os.path.join(output_path, "summary_statistics_KNN.txt"), sep='\t', index=False)

✅ TIFF guardado correctamente: Z:\users\6331823\Mario Pipeline\frames-frames_extracted\Second try\cellpose\distances\Channel_stack_TZCYX.tiff


Make figures

In [ ]:
# Asegurar tipo correcto
summary_df['frame'] = summary_df['frame'].astype(int)

# Agregar %WT
summary_df['pct_WT'] = summary_df['WT_cells'] / summary_df['total_cells'] * 100

# Calcular medias por frame
mean_df = summary_df.groupby('frame').agg({
    'median_knn_dist': 'mean',
    'pct_WT': 'mean'
}).reset_index()

# --- PLOT Median KNN ---
plt.figure(figsize=(10, 5))

# Líneas por series_id
sns.lineplot(data=summary_df, x='frame', y='median_knn_dist', hue='series_id',
             palette='tab10', linewidth=1, alpha=0.5, legend=False)

# Dots por frame y TL
sns.stripplot(data=summary_df, x='frame', y='median_knn_dist', hue='series_id',
              dodge=True, jitter=False, alpha=0.5, size=6, palette='tab10', legend=False)


plt.title("Median KNN distance per frame")
plt.xlabel("Frame")
plt.ylabel("Median KNN distance")
plt.legend(title='TL (series_id)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# --- PLOT % WT ---
plt.figure(figsize=(10, 5))

# Líneas por series_id
sns.lineplot(data=summary_df, x='frame', y='pct_WT', hue='series_id',
             palette='tab10', linewidth=1, alpha=0.5, legend=False)

# Dots por frame y TL
sns.stripplot(data=summary_df, x='frame', y='pct_WT', hue='series_id',
              dodge=True, jitter=False, alpha=0.5, size=6, palette='tab10', legend=False)

plt.title("% WT cells per frame")
plt.xlabel("Frame")
plt.ylabel("Percentage of WT cells")
plt.legend(title='TL (series_id)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()